# C01 · Data validation with `bench validate` (checks 1–4)

**Task 2 of the next-steps guide.** Checks that the new package reproduces the 3 Nov Colab results
(`bright_event_summary.csv`). This is also the verification step for the AI-written code
(record the outcome in `AI_USE_LOG.md`).

Before the freeze, confirmatory events show only a schema check, not class counts.

In [ ]:
# 1. Get the code. The repo is private: add a GitHub token (read access to this repo) in
#    Colab -> Secrets (key icon on the left) with the name GH_TOKEN, and enable notebook access.
from google.colab import userdata
GH = userdata.get("GH_TOKEN")
REPO = f"https://{GH}@github.com/ThinukaLehan/damagesar-bench.git"
!git clone -q $REPO /content/damagesar-bench 2>/dev/null || (cd /content/damagesar-bench && git pull -q)
%cd /content/damagesar-bench
!pip install -q -e .
# A running Colab kernel does not see an editable install until a restart: put src/ on the path.
import sys, importlib
if "/content/damagesar-bench/src" not in sys.path:
    sys.path.insert(0, "/content/damagesar-bench/src")
importlib.invalidate_caches()
!bench status

In [ ]:
# Download BRIGHT SAR + labels from Hugging Face straight to /content (~3.4 GB, ~1 min).
# Copying from Drive fails (Errno 107), so always download fresh.
import os, zipfile
from huggingface_hub import hf_hub_download
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass  # BRIGHT is not gated; the token only avoids rate limits

ROOT = "/content/bright"
os.makedirs(ROOT, exist_ok=True)
for name in ["post-event.zip", "target.zip"]:
    folder = os.path.join(ROOT, name.replace(".zip", ""))
    if os.path.isdir(folder) and len(os.listdir(folder)) > 4000:
        print("already there:", folder); continue
    p = hf_hub_download("Kullervo/BRIGHT", name, repo_type="dataset", local_dir="/content/zips")
    with zipfile.ZipFile(p) as z:
        z.extractall(ROOT)
    print("unzipped", name)
for sub in ["post-event", "target"]:
    print(sub, len(os.listdir(os.path.join(ROOT, sub))), "files")

In [ ]:
# 2. Validate (takes a few minutes: every tile is opened once)
from google.colab import drive
drive.mount("/content/drive")
from damagesar_bench import drive as layout
layout.ensure_layout()
OUT = str(layout.run_dir("c01_data_validation"))
# names used by the final cell that files the executed notebook
import os, datetime
NOTEBOOK_NAME, RUN, PROJECT = "c01_data_validation", OUT, str(layout.project_root())
print("This run saves to:", OUT)
!bench validate --root /content/bright --out $OUT
!bench audit --out $OUT

In [ ]:
# 3. Compare with the 3 Nov summary (tile counts per event should match exactly)
import os, pandas as pd
new = pd.read_csv(f"{OUT}/event_summary.csv").set_index("event")
# The 3 Nov summary: copy it from the old account into 01_data/derived/ first
old_path = f"{layout.path('derived')}/bright_event_summary.csv"
if not os.path.exists(old_path):
    # Do not stop the notebook: the final cell must still save the executed copy.
    print("SKIPPED: 3 Nov summary not found at", old_path)
    print("Copy bright_event_summary.csv from the old account into 01_data/derived/, then rerun this cell.")
else:
    old = pd.read_csv(old_path)
    print("old columns:", list(old.columns))
    # Adjust 'event' / 'n_tiles' below to the old file's column names if they differ.
    ev_col = "event" if "event" in old.columns else old.columns[0]
    tile_col = next((c for c in old.columns if "tile" in c.lower()), None)
    cmp = new[["role", "n_tiles", "n_excluded_nodata"]].join(old.set_index(ev_col)[[tile_col]], how="outer")
    cmp["match"] = cmp["n_tiles"] == cmp[tile_col]
    display(cmp)
    print("ALL MATCH" if cmp["match"].all() else "MISMATCH: check before trusting the package")
    cmp.to_csv(f"{OUT}/compare_with_3nov.csv")

In [ ]:
# File the executed notebook (with outputs) into this run's folder and 03_notebooks/checks/
import json
from google.colab import _message
# Ask the browser for the notebook; it can time out (busy tab, large outputs). Results above are already saved.
nb = None
for attempt in range(3):
    try:
        reply = _message.blocking_request("get_ipynb", timeout_sec=120)
        nb = (reply or {}).get("ipynb")
    except Exception as ex:
        print("attempt", attempt + 1, "failed:", ex)
    if nb:
        break
if not nb:
    print("Could not copy the executed notebook automatically (the results above are already saved).")
    print("Use File -> Save a copy in Drive, then move the copy into", RUN)
else:
    stamp = datetime.date.today().isoformat()
    for target in (f"{RUN}/{NOTEBOOK_NAME}_executed.ipynb",
                   f"{PROJECT}/03_notebooks/checks/{stamp}_{NOTEBOOK_NAME}.ipynb"):
        with open(target, "w") as fh:
            json.dump(nb, fh)
        print("saved", target)
    print("run folder:", sorted(os.listdir(RUN)))
